# log-analytics-engine — Colab Benchmark
**Numba parallel parser + JAX vectorised aggregation vs pandas baseline**

Run cells top to bottom. Results are saved to `results/benchmark.json` and downloaded at the end.

In [ ]:
# ── 1. Clone repo & install deps ──────────────────────────────────────────
!git clone https://github.com/pradneshfernandez/log-analytics-engine
%cd log-analytics-engine
!pip install numba "jax[cuda12]" numpy pandas -q

In [ ]:
# ── 2. Generate 1 GB synthetic log file ───────────────────────────────────
!python data/generate_logs.py --output data/1gb.log --size-gb 1

In [ ]:
# ── 3. Verify file ────────────────────────────────────────────────────────
import os
size_mb = os.path.getsize('data/1gb.log') / 1024**2
print(f'File size: {size_mb:.1f} MB')
!head -3 data/1gb.log

## Full pipeline run (parse → aggregate → anomaly detect)

In [ ]:
# ── 4. Run full pipeline & print summary report ───────────────────────────
import sys
sys.path.insert(0, '.')

from src.pipeline import run
summary = run('data/1gb.log')

## Benchmarks — pandas vs Numba vs Numba+JAX

In [ ]:
# ── 5. Pandas baseline ────────────────────────────────────────────────────
import re, time, tracemalloc
import numpy as np
import pandas as pd
from pathlib import Path

_CLF_RE = re.compile(
    r'^(\S+) \S+ \S+ \[([^\]]+)\] "(\S+) (\S+) \S+" (\d+) (\d+)'
)

LOG_PATH = 'data/1gb.log'

print('Running pandas benchmark...')
tracemalloc.start()
t0 = time.perf_counter()

rows = []
with open(LOG_PATH, encoding='ascii', errors='ignore') as fh:
    for line in fh:
        m = _CLF_RE.match(line)
        if m:
            ip, ts, method, endpoint, status, size = m.groups()
            rows.append((ip, ts, endpoint, int(status), int(size)))

df = pd.DataFrame(rows, columns=['ip','timestamp','endpoint','status','size'])
_rpm        = df.groupby('timestamp')['status'].count()
_err_rate   = (df['status'] >= 400).mean()
_pcts       = np.percentile(df['size'], [50, 95, 99])
_top10      = df.groupby('endpoint')['status'].count().nlargest(10)

pandas_sec  = time.perf_counter() - t0
_, peak     = tracemalloc.get_traced_memory()
tracemalloc.stop()

pandas_lines   = len(df)
pandas_lps     = pandas_lines / pandas_sec
pandas_mem_mb  = peak / 1024**2

print(f'  lines      : {pandas_lines:,}')
print(f'  time       : {pandas_sec:.2f}s')
print(f'  lines/sec  : {pandas_lps:,.0f}')
print(f'  peak mem   : {pandas_mem_mb:.0f} MB')
print(f'  error rate : {_err_rate*100:.2f}%')
print(f'  p50/p95/p99: {_pcts[0]:.0f} / {_pcts[1]:.0f} / {_pcts[2]:.0f} bytes')

In [ ]:
# ── 6. Numba parser only ──────────────────────────────────────────────────
from src.parser import parse_file

print('Warming up Numba JIT (first call compiles, not counted)...')
parse_file(LOG_PATH)
print('done')

print('Running Numba benchmark...')
tracemalloc.start()
t0 = time.perf_counter()

parsed = parse_file(LOG_PATH)

numba_sec  = time.perf_counter() - t0
_, peak    = tracemalloc.get_traced_memory()
tracemalloc.stop()

numba_lines  = len(parsed['timestamps'])
numba_lps    = numba_lines / numba_sec
numba_mem_mb = peak / 1024**2

print(f'  lines     : {numba_lines:,}')
print(f'  time      : {numba_sec:.2f}s')
print(f'  lines/sec : {numba_lps:,.0f}')
print(f'  peak mem  : {numba_mem_mb:.0f} MB')

In [ ]:
# ── 7. Full pipeline: Numba parse + JAX aggregate + anomaly detect ─────────
import jax
import jax.numpy as jnp
from src.aggregator import compute_all_metrics, compute_n_windows, top_k_endpoints
from src.anomaly    import detect_anomalies, extract_anomaly_events

device = jax.default_backend()
print(f'JAX backend: {device.upper()}')

# Warm-up
print('Warming up Numba+JAX JIT...')
_p  = parse_file(LOG_PATH)
_ts = np.sort(_p['timestamps'])
_nw = compute_n_windows(_ts)
_r  = compute_all_metrics(
    jnp.array(_ts), jnp.array(_p['status_codes']),
    jnp.array(_p['response_sizes']), jnp.array(_p['endpoint_hashes']),
    n_windows=_nw
)
_r.rpm.block_until_ready()
detect_anomalies(_r.rpm, int(_ts[0]))[0].block_until_ready()
print('done')

# Timed run
print('Running full pipeline benchmark...')
tracemalloc.start()
t0 = time.perf_counter()

parsed     = parse_file(LOG_PATH)
sort_idx   = np.argsort(parsed['timestamps'])
timestamps = parsed['timestamps'][sort_idx]
status     = parsed['status_codes'][sort_idx]
sizes      = parsed['response_sizes'][sort_idx]
ep_hashes  = parsed['endpoint_hashes'][sort_idx]

n_wins = compute_n_windows(timestamps)
result = compute_all_metrics(
    jnp.array(timestamps), jnp.array(status),
    jnp.array(sizes), jnp.array(ep_hashes),
    n_windows=n_wins
)
result.rpm.block_until_ready()

mask, atimes, sev = detect_anomalies(result.rpm, int(timestamps[0]))
mask.block_until_ready()

full_sec  = time.perf_counter() - t0
_, peak   = tracemalloc.get_traced_memory()
tracemalloc.stop()

full_lines  = len(timestamps)
full_lps    = full_lines / full_sec
full_mem_mb = peak / 1024**2
events      = extract_anomaly_events(np.asarray(mask), np.asarray(atimes), np.asarray(sev))

print(f'  lines      : {full_lines:,}')
print(f'  time       : {full_sec:.2f}s')
print(f'  lines/sec  : {full_lps:,.0f}')
print(f'  peak mem   : {full_mem_mb:.0f} MB')
print(f'  anomalies  : {len(events)}')
print(f'  p50/p95/p99: {int(result.p50)} / {int(result.p95)} / {int(result.p99)} bytes')

## Results comparison table

In [ ]:
# ── 8. Print comparison table & compute speed-ups ─────────────────────────
file_mb = os.path.getsize(LOG_PATH) / 1024**2

rows_table = [
    ('pandas',                 pandas_lines, pandas_sec, pandas_lps, pandas_mem_mb),
    ('numba (parse only)',     numba_lines,  numba_sec,  numba_lps,  numba_mem_mb),
    (f'numba+jax ({device})', full_lines,   full_sec,   full_lps,   full_mem_mb),
]

col = '{:<22} {:>10} {:>10} {:>14} {:>10}'
sep = '-' * 70
print(f'\nFile: {file_mb:.0f} MB  (~{pandas_lines:,} lines)\n')
print(sep)
print(col.format('Engine', 'Lines', 'Time(s)', 'Lines/s', 'Mem(MB)'))
print(sep)
for name, n, t, lps, mem in rows_table:
    print(col.format(name, f'{n:,}', f'{t:.2f}', f'{lps:,.0f}', f'{mem:.0f}'))
print(sep)

print('\nSpeed-up vs pandas:')
for name, _, _, lps, _ in rows_table[1:]:
    print(f'  {name:<22}  {lps/pandas_lps:.1f}×')

In [ ]:
# ── 9. Save results to JSON & download ────────────────────────────────────
import json
from datetime import datetime, timezone

os.makedirs('results', exist_ok=True)

results_data = {
    'run_at':      datetime.now(tz=timezone.utc).isoformat(),
    'jax_backend': device,
    'file_mb':     round(file_mb, 1),
    'results': [
        {'engine': 'pandas',                'n_lines': pandas_lines, 'elapsed_sec': round(pandas_sec, 3), 'lines_per_sec': round(pandas_lps), 'peak_mem_mb': round(pandas_mem_mb)},
        {'engine': 'numba_parse_only',      'n_lines': numba_lines,  'elapsed_sec': round(numba_sec, 3),  'lines_per_sec': round(numba_lps),  'peak_mem_mb': round(numba_mem_mb)},
        {'engine': f'numba_jax_{device}',   'n_lines': full_lines,   'elapsed_sec': round(full_sec, 3),   'lines_per_sec': round(full_lps),   'peak_mem_mb': round(full_mem_mb),
         'p50_bytes': int(result.p50), 'p95_bytes': int(result.p95), 'p99_bytes': int(result.p99),
         'anomalies_detected': len(events)},
    ],
    'speedups': [
        {'engine': 'numba_parse_only',    'vs_pandas_x': round(numba_lps / pandas_lps, 2)},
        {'engine': f'numba_jax_{device}', 'vs_pandas_x': round(full_lps  / pandas_lps, 2)},
    ]
}

with open('results/benchmark.json', 'w') as f:
    json.dump(results_data, f, indent=2)

print('Saved to results/benchmark.json')
print(json.dumps(results_data, indent=2))

# Download the file
from google.colab import files
files.download('results/benchmark.json')